# 8.10 評分者真的分得出目標風格嗎？

# 第 8 章：風格、個性與指令遵循

前置：第7章SFT。把風格與內容分開想：想像同一位老師可以說得生動，也可以精準遵循格式。本章先用人工編寫、內容固定的短回答校準目標；LoRA是後半支線。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：有個性也要可靠：三個分開的面向**

漂亮文筆不能替代正確答案或格式遵循。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/style.svg")))

**先想一想：**裁判每題都給高分，平均分高代表可靠嗎？

自動評分先用人工已知正負例校準。若它把冗長錯誤答案當佳作，先修評分規則，再讓它給大批結果打分。

**把直覺寫成數學：**agreement=與人工一致的筆數/總筆數；小樣本只能發現明顯偏差。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
human = torch.tensor([1, 0, 1, 0])
recorded_judge = torch.tensor([1, 1, 1, 0])
print("人工/預錄裁判一致", (human == recorded_judge).float().mean().item())
print("需要回讀的分歧例", (human != recorded_judge).nonzero().flatten().tolist())

**如何讀結果：**預錄分數只為說明流程，不聲稱實際API評測。保留原回答供人核對。

**只改一件事：**只把一個人評正例改成負例，重新討論規則。
